Copyright **`(c)`** 2025 Giovanni Squillero `<giovanni.squillero@polito.it>`  
[`https://github.com/squillero/computational-intelligence`](https://github.com/squillero/computational-intelligence)  
Free under certain conditions — see the [`license`](https://github.com/squillero/computational-intelligence/blob/master/LICENSE.md) for details.  

In [7]:
from itertools import product, combinations
import numpy as np
import networkx as nx
from icecream import ic

In [8]:
def create_problem(
    size: int,
    *,
    density: float = 1.0,
    negative_values: bool = False,
    noise_level: float = 0.0,
    seed: int = 42,
) -> np.ndarray:
    """Problem generator for Lab3"""
    rng = np.random.default_rng(seed)
    # Genera N (size) punti casuali nel piano
    map = rng.random(size=(size, 2))
    # Inizializza la matrice dei pesi casuali
    problem = rng.random((size, size))
    if negative_values:
        problem = problem * 2 - 1
    problem *= noise_level
    for a, b in product(range(size), repeat=2):
        if rng.random() < density:
            # se l'arco esiste distanza euclidea + rumore (dell'inizializzazione)
            problem[a, b] += np.sqrt(
                np.square(map[a, 0] - map[b, 0]) + np.square(map[a, 1] - map[b, 1])
            )
        else:
            # se l'arco non esiste peso infinito
            problem[a, b] = np.inf
    np.fill_diagonal(problem, 0)
    return (problem * 1_000).round()


# Crea N punti in un piano 2D.
#Calcola le loro distanze euclidee.
#Usa queste distanze come pesi degli archi del grafo.
#Con density < 1, alcuni archi vengono rimossi (posti a inf → nessun collegamento).

#Aggiunge eventualmente rumore e valori negativi.

In [9]:
problem = create_problem(10, density=0.15, noise_level=10, negative_values=True)
problem

array([[    0.,    inf, 10280.,    inf,  6214.,    inf,    inf, -8731.,
        -6480.,    inf],
       [   inf,     0.,    inf,    inf,    inf, -5672.,    inf,    inf,
           inf,  3469.],
       [   inf,    inf,     0.,    inf,    inf,    inf,    inf,    inf,
           inf, -6395.],
       [   inf,    inf,  6432.,     0.,  4820.,    inf,    inf,    inf,
           inf,    inf],
       [ 4014.,    inf,    inf,  6016.,     0.,    inf,    inf,    inf,
           inf,    inf],
       [   inf,    inf,    inf,    inf,    inf,     0.,    inf,    inf,
           inf,    inf],
       [   inf,    inf,    inf,    inf,    inf,    inf,     0.,  -134.,
           inf,    inf],
       [   inf,    inf,  -251.,    inf,    inf,    inf,    inf,     0.,
           inf,    inf],
       [   inf,    inf,    inf, 10135.,  6151.,    inf,  -249.,    inf,
            0.,    inf],
       [   inf,    inf,    inf,    inf,    inf,    inf,  5436.,  4947.,
           inf,     0.]])

In [11]:
masked = np.ma.masked_array(problem, mask=np.isinf(problem))
G = nx.from_numpy_array(masked, create_using=nx.DiGraph)

In [ ]:
for s, d in combinations(range(problem.shape[0]), 2):
    try:
        # path = nx.shortest_path(G, s, d, weight='weight')
        path = nx.bellman_ford_path(G, s, d, weight='weight')
        cost = cost = nx.path_weight(G, path, weight='weight')
    except nx.NetworkXNoPath:
        # Nodes are not connected
        path = None
        cost = np.inf
    except nx.NetworkXUnbounded:
        # Negative cycle detected
        path = None
        cost = -np.inf
    ic(s, d, path, cost)
None

In [16]:
#Best-first search implementation
##Uniform-cost search  - - - 
# Like Breadth-first, but the node with the lowest path cost (lower cost) from the root is expanded, 
# The frontier is a real priority queue, first node expanded is the closest from the actual node, 
# Also called Dijkstra’s algorithm. 

def uniform_cost_search(graph: np.ndarray, start: int, goal: int) -> tuple[list[int], float]:
    
    size = graph.shape[0]
    visited = [False] * size
    parent = [None] * size
    # ( nodo, costo accumulato )
    frontier = [(start, 0.0)]
    
    visited[start] = True

    while frontier:
        ##ic(frontier)
        current_node, current_cost = frontier.pop(0)

        if current_node == goal:
            path = []
            while current_node is not None:
                path.append(current_node)
                current_node = parent[current_node]
            return path[::-1], current_cost

        for neighbor in range(size):
            weight = graph[current_node, neighbor]
            if weight != np.inf and not visited[neighbor]:
                visited[neighbor] = True
                parent[neighbor] = current_node
                
                frontier.append((neighbor, current_cost + weight))
        # ordina la coda in base al costo accumulato (crescente)
        frontier.sort(key=lambda x: x[1])  # best-first
        

    return None, np.inf


s, d = 1, 2
path, cost = uniform_cost_search(problem, s, d)
ic(s, d, path, cost)

ic| s: 1, d: 2, path: [1, 9, 7, 2], cost: np.float64(8165.0)


(1, 2, [1, 9, 7, 2], np.float64(8165.0))

In [ ]:
import numpy as np

def best_fit(graph: np.ndarray, start: int, goal: int) -> tuple[list[int], float]:
    
    size = graph.shape[0]
    # ( nodo, costo accumulato )
    
    #esclude percorsi circolari
    # mantiene per ogni percorso i nodi visitati
    feasible_paths=[]
    frontier=[]
    path_id = 0
    for i in range(size):
        if i != start and graph[start, i] != np.inf:
            frontier.append((path_id, i, graph[start, i]))
            feasible_paths.append([start, i])
            path_id += 1
    
    #print("frontiera iniziale:", frontier)
    #print("path possibili iniziali: ", feasible_paths)
    goal_path=[]    

    while frontier:
        #print ("frontiera:", frontier)
        path_id, current_node, current_cost = frontier.pop(0)
        #print ("estraggo:", path_id, current_node, current_cost)
        if current_node == goal:
            
            path= feasible_paths[path_id]
            #print("goal reached", path)
            if path not in [p[0] for p in goal_path]:
             goal_path.append((path, current_cost))
             #print(path_id, path, current_cost)
            continue
            

        for neighbor in range(size):
            weight = graph[current_node, neighbor]
            if weight != np.inf and neighbor not in feasible_paths[path_id]:
                current_path= feasible_paths[path_id]
                new_path=current_path + [neighbor]
                ## evita percorsi uguali
                if new_path not in [p for p in feasible_paths]:
                    ##add new path
                    new_path_id= len(feasible_paths)
                    feasible_paths.append(new_path)

                    #print(new_path_id, feasible_paths[new_path_id])
                    frontier.append((new_path_id, neighbor, current_cost + weight))
                #else:
                    #print("ignoro nodo:", neighbor, " path:", current_path)
            #else:
                #print("ignoro nodo:", neighbor, "weight:", weight)
                
        # ordina la coda in base al costo accumulato (crescente)
        frontier.sort(key=lambda x: x[1])  # best-first
        
    if goal_path:
        return min(goal_path, key=lambda x: x[1])
       
    return None, np.inf


s, d = 1, 2
path, cost = best_fit(problem, s, d)
print( "start:", s, " end:", d, " path:", path, " cost:", cost)

frontiera iniziale: [(0, 5, np.float64(-5672.0)), (1, 9, np.float64(3469.0))]
path possibili iniziali:  [[1, 5], [1, 9]]
frontiera: [(0, 5, np.float64(-5672.0)), (1, 9, np.float64(3469.0))]
estraggo: 0 5 -5672.0
ignoro nodo: 0 weight: inf
ignoro nodo: 1 weight: inf
ignoro nodo: 2 weight: inf
ignoro nodo: 3 weight: inf
ignoro nodo: 4 weight: inf
ignoro nodo: 5 weight: 0.0
ignoro nodo: 6 weight: inf
ignoro nodo: 7 weight: inf
ignoro nodo: 8 weight: inf
ignoro nodo: 9 weight: inf
frontiera: [(1, 9, np.float64(3469.0))]
estraggo: 1 9 3469.0
ignoro nodo: 0 weight: inf
ignoro nodo: 1 weight: inf
ignoro nodo: 2 weight: inf
ignoro nodo: 3 weight: inf
ignoro nodo: 4 weight: inf
ignoro nodo: 5 weight: inf
2 [1, 9, 6]
3 [1, 9, 7]
ignoro nodo: 8 weight: inf
ignoro nodo: 9 weight: 0.0
frontiera: [(2, 6, np.float64(8905.0)), (3, 7, np.float64(8416.0))]
estraggo: 2 6 8905.0
ignoro nodo: 0 weight: inf
ignoro nodo: 1 weight: inf
ignoro nodo: 2 weight: inf
ignoro nodo: 3 weight: inf
ignoro nodo: 4 weigh

In [22]:
# Depth-first

def dfs(graph: np.ndarray, start: int, goal: int) -> tuple[list[int], float]:
    
    size = graph.shape[0]
    visited = [False] * size
    parent = [None] * size
    # ( nodo, costo accumulato )
    stack = [(start, 0.0)]
    
    visited[start] = True

    while stack:
       # ic(stack)
        current_node, current_cost = stack.pop()

        if current_node == goal:
            path = []
            while current_node is not None:
                path.append(current_node)
                current_node = parent[current_node]
            return path[::-1], current_cost

        for neighbor in range(size):
            weight = graph[current_node, neighbor]
            if weight != np.inf and not visited[neighbor]:
                visited[neighbor] = True
                parent[neighbor] = current_node
                
                stack.append((neighbor, current_cost + weight))
        

    return None, np.inf

s, d = 1, 2
path, cost = dfs(problem, s, d)
ic(s, d, path, cost)


ic| s: 1
    d: 2
    path: [1, 47, 40, 43, 38, 20, 44, 2]
    cost: np.float64(20264.0)


(1, 2, [1, 47, 40, 43, 38, 20, 44, 2], np.float64(20264.0))

In [17]:

for s, d in combinations(range(problem.shape[0]), 2):
    p1, c1 =best_fit(problem, s, d)
    p2, c2 =uniform_cost_search(problem, s, d)
    if p1 != p2 or c1 != c2:
        print("Discrepancy found:")
        ic(s, d)
        ic("Best-fit:", p1, c1)
        ic("uniform cost:", p2, c2)
    # else:
    #     ic(s, d, p1, c1)
None

ic| s: 0, d: 2
ic| 'Best-fit:',

 p1: [0, 7, 2], c1: np.float64(-8982.0)
ic| 'uniform cost:', p2: [0, 2], c2: np.float64(10280.0)
ic| s: 0, d: 4
ic| 'Best-fit:', p1: [0, 8, 4], c1: np.float64(-329.0)
ic| 'uniform cost:', p2: [0, 4], c2: np.float64(6214.0)
ic| s: 0, d: 6
ic| 'Best-fit:', p1: [0, 7, 2, 9, 6], c1: np.float64(-9941.0)
ic| 'uniform cost:', p2: [0, 8, 6], c2: np.float64(-6729.0)


frontiera iniziale: [(0, 2, np.float64(10280.0)), (1, 4, np.float64(6214.0)), (2, 7, np.float64(-8731.0)), (3, 8, np.float64(-6480.0))]
path possibili iniziali:  [[0, 2], [0, 4], [0, 7], [0, 8]]
frontiera: [(0, 2, np.float64(10280.0)), (1, 4, np.float64(6214.0)), (2, 7, np.float64(-8731.0)), (3, 8, np.float64(-6480.0))]
estraggo: 0 2 10280.0
ignoro nodo: 0 weight: inf
ignoro nodo: 1 weight: inf
ignoro nodo: 2 weight: 0.0
ignoro nodo: 3 weight: inf
ignoro nodo: 4 weight: inf
ignoro nodo: 5 weight: inf
ignoro nodo: 6 weight: inf
ignoro nodo: 7 weight: inf
ignoro nodo: 8 weight: inf
4 [0, 2, 9]
frontiera: [(1, 4, np.float64(6214.0)), (2, 7, np.float64(-8731.0)), (3, 8, np.float64(-6480.0)), (4, 9, np.float64(3885.0))]
estraggo: 1 4 6214.0
ignoro nodo: 0 weight: 4014.0
ignoro nodo: 1 weight: inf
ignoro nodo: 2 weight: inf
5 [0, 4, 3]
ignoro nodo: 4 weight: 0.0
ignoro nodo: 5 weight: inf
ignoro nodo: 6 weight: inf
ignoro nodo: 7 weight: inf
ignoro nodo: 8 weight: inf
ignoro nodo: 9 weight: 

ic| s: 0, d: 9
ic| 'Best-fit:', p1: [0, 7, 2, 9], c1: np.float64(-15377.0)
ic| 'uniform cost:', p2: [0, 2, 9], c2: np.float64(3885.0)
ic| s: 3, d: 6
ic| 'Best-fit:', p1: [3, 4, 0, 7, 2, 9, 6], c1: np.float64(-1107.0)
ic| 'uniform cost:', p2: [3, 2, 9, 6], c2: np.float64(5473.0)
ic| s: 3, d: 7
ic| 'Best-fit:', p1: [3, 4, 0, 7], c1: np.float64(103.0)
ic| 'uniform cost:', p2: [3, 2, 9, 7], c2: np.float64(4984.0)
ic| s: 3, d: 9
ic| 'Best-fit:', p1: [3,

 inf
17 [0, 8, 4, 3, 2]
ignoro nodo: 3 weight: 0.0
ignoro nodo: 4 weight: 4820.0
ignoro nodo: 5 weight: inf
ignoro nodo: 6 weight: inf
ignoro nodo: 7 weight: inf
ignoro nodo: 8 weight: inf
ignoro nodo: 9 weight: inf
frontiera: [(17, 2, np.float64(12119.0)), (14, 4, np.float64(8475.0)), (12, 6, np.float64(-6729.0)), (4, 9, np.float64(3885.0)), (7, 9, np.float64(12267.0)), (9, 9, np.float64(-15377.0)), (15, 9, np.float64(3692.0))]
estraggo: 17 2 12119.0
ignoro nodo: 0 weight: inf
ignoro nodo: 1 weight: inf
ignoro nodo: 2 weight: 0.0
ignoro nodo: 3 weight: inf
ignoro nodo: 4 weight: inf
ignoro nodo: 5 weight: inf
ignoro nodo: 6 weight: inf
ignoro nodo: 7 weight: inf
ignoro nodo: 8 weight: inf
18 [0, 8, 4, 3, 2, 9]
frontiera: [(14, 4, np.float64(8475.0)), (12, 6, np.float64(-6729.0)), (4, 9, np.float64(3885.0)), (7, 9, np.float64(12267.0)), (9, 9, np.float64(-15377.0)), (15, 9, np.float64(3692.0)), (18, 9, np.float64(5724.0))]
estraggo: 14 4 8475.0
ignoro nodo: 0 weight: 4014.0
ignoro nodo

 4, 0, 7, 2, 9], c1: np.float64(-6543.0)
ic| 'uniform cost:', p2: [3, 2, 9], c2: np.float64(37.0)
ic| s: 4, d: 6
ic| 'Best-fit:', p1: [4, 0, 7, 2, 9, 6], c1: np.float64(-5927.0)
ic| 'uniform cost:', p2: [4, 0, 8, 6], c2: np.float64(-2715.0)
ic| s: 4, d: 9
ic| 'Best-fit:', p1: [4, 0, 7, 2, 9], c1: np.float64(-11363.0)
ic| 'uniform cost:', p2: [4, 0, 2, 9], c2: np.float64(7899.0)


frontiera iniziale: [(0, 0, np.float64(4014.0)), (1, 3, np.float64(6016.0))]
path possibili iniziali:  [[4, 0], [4, 3]]
frontiera: [(0, 0, np.float64(4014.0)), (1, 3, np.float64(6016.0))]
estraggo: 0 0 4014.0
ignoro nodo: 0 weight: 0.0
ignoro nodo: 1 weight: inf
2 [4, 0, 2]
ignoro nodo: 3 weight: inf
ignoro nodo: 4 weight: 6214.0
ignoro nodo: 5 weight: inf
ignoro nodo: 6 weight: inf
3 [4, 0, 7]
4 [4, 0, 8]
ignoro nodo: 9 weight: inf
frontiera: [(2, 2, np.float64(14294.0)), (1, 3, np.float64(6016.0)), (3, 7, np.float64(-4717.0)), (4, 8, np.float64(-2466.0))]
estraggo: 2 2 14294.0
ignoro nodo: 0 weight: inf
ignoro nodo: 1 weight: inf
ignoro nodo: 2 weight: 0.0
ignoro nodo: 3 weight: inf
ignoro nodo: 4 weight: inf
ignoro nodo: 5 weight: inf
ignoro nodo: 6 weight: inf
ignoro nodo: 7 weight: inf
ignoro nodo: 8 weight: inf
5 [4, 0, 2, 9]
frontiera: [(1, 3, np.float64(6016.0)), (3, 7, np.float64(-4717.0)), (4, 8, np.float64(-2466.0)), (5, 9, np.float64(7899.0))]
estraggo: 1 3 6016.0
ignoro no